# SPY capture: a first look at the database

This notebook inventories the collected data and walks through representative records. It does **not** yet certify the collection against the BSM handoff or validate the solver.

Before running:
- Stop the collector. All connections here are **read-only** and close after each query.
- Select the standard-Python **`.venv-collector`** kernel, not the free-threaded demo environment.
- To retain results privately, save a working copy under `Tastyware_Demo_Scripts/exports/notebooks/` before running. That directory is Git-ignored. Keep this source notebook free of saved outputs.
- Run cells in order. Queries aggregate inside DuckDB or fetch small previews; they do not load the complete event history into pandas.

Times are displayed in UTC unless a column explicitly ends in `_et` (America/New_York).

## 1. Locate and open the capture safely

The default is the collector's existing database, not its pre-upgrade backup. You can replace `DATABASE_PATH` with another explicit path. A missing file raises an error rather than creating an empty database. This first version expects collector schema 2, which can also contain earlier schema-1 runs.

In [ ]:
from pathlib import Path
import json
import sys

import duckdb
import pandas as pd
from IPython.display import display

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Tastyware_Demo_Scripts" / "collector_core.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from inside the Tasty_Dearcyfi_Tests repository.")

DATABASE_PATH = repo_root / "Tastyware_Demo_Scripts" / "exports" / "spy_validation.duckdb"
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f"Capture database does not exist: {DATABASE_PATH}")

def query(sql: str, parameters: list[object] | None = None) -> pd.DataFrame:
    with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
        connection.execute("SET TimeZone='UTC'")
        return connection.execute(sql, parameters if parameters is not None else []).df()

schema = query("SELECT version FROM collector_schema")
if schema["version"].tolist() != [2]:
    raise ValueError(f"Expected collector schema 2; found {schema['version'].tolist()}.")

pd.set_option("display.max_columns", 24)
pd.set_option("display.max_colwidth", 100)
wal_path = Path(str(DATABASE_PATH) + ".wal")
print(f"Python: {sys.executable}")
print(f"DuckDB: {duckdb.__version__}; collector schema: 2")
print(f"Database: {DATABASE_PATH}")
print(f"Database size: {DATABASE_PATH.stat().st_size / 1_000_000_000:.3f} decimal GB")
print(f"WAL present: {wal_path.exists()}")
print("Read-only access; every query closes its connection.")

## 2. What tables and views exist?

Counts below are exact for base tables. Views are listed without counting them again because they derive from the same underlying records. **Do not add all table counts together as a count of unique market events.**

| Layer | Main relations | Meaning |
|---|---|---|
| Provenance | `runs`, `instruments`, `universe_revisions` | Execution history, contract terms, and subscription selections |
| Received updates | `raw_updates`, `raw_update_payloads`, `raw_records` | One delivery row per received update; the payload view joins content-addressed JSON |
| Normalized observations | `quotes`, `trades`, `analytics`, `candles` | Numeric fields and timestamps; schema-1 history was sampled, schema-2 history records received updates |
| Minute sampling | `cycles`, `samples` | Scheduled sampling attempts and references to selected observations; not time-window averages |
| Operational context | `requests`, `diagnostics`, `run_coverage` | Requests, gaps, warnings, and summary counts |

`latest_candles` is a latest-revision view, not a promise of provider finality.

In [ ]:
relations = query("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema='main' AND table_catalog=current_database()
    ORDER BY table_name
""")
row_counts = []
for relation in relations.itertuples(index=False):
    if relation.table_type == "BASE TABLE":
        identifier = '"' + relation.table_name.replace('"', '""') + '"'
        row_counts.append(int(query(f"SELECT count(*) AS rows FROM {identifier}")["rows"].iloc[0]))
    else:
        row_counts.append(pd.NA)
relations["base_table_rows"] = pd.array(row_counts, dtype="Int64")
display(relations)

### Inspect the columns of one relation

Change `RELATION_TO_INSPECT` to a name from the inventory, such as `samples`, `quotes`, or `analytics`. `record_id` identifies a row; `raw_record_id` links to retained JSON. A received update's `observation_id` links to its normalized observation.

In [ ]:
RELATION_TO_INSPECT = "raw_updates"
if RELATION_TO_INSPECT not in set(relations["table_name"]):
    raise ValueError(f"Unknown relation: {RELATION_TO_INSPECT}")
identifier = '"' + RELATION_TO_INSPECT.replace('"', '""') + '"'
display(query(f"DESCRIBE {identifier}"))

## 3. Which collection runs are present?

A restart appends a new run; it does not overwrite previous runs. This database may include failed startup attempts, short diagnostic captures, the earlier minute-only collector, and the longer unattended capture.

The recording mode is taken from each run's saved capabilities. Schema-1 runs do not have complete raw-update history, even after the database is upgraded. `partial` is a coverage/status result, not proof that the captured rows are unusable; `completed` would not certify BSM suitability either.

In [ ]:
runs = query("""
    SELECT r.run_id, r.schema_version, r.session_date,
           r.started_at, r.ended_at, r.status, r.stop_reason,
           json_extract_string(r.capabilities, '$.recording_mode') AS recording_mode,
           coalesce(u.received_updates, 0) AS received_updates
    FROM runs r
    LEFT JOIN (
        SELECT run_id, count(*) AS received_updates FROM raw_updates GROUP BY run_id
    ) u USING (run_id)
    ORDER BY received_updates DESC, r.started_at DESC
""")
if runs.empty:
    raise ValueError("The database contains no collection runs.")
display(runs)

### Choose a run for the remaining cells

By default, select the run with the largest received-update count. Replace the assignment with a specific run ID to examine another attempt. The rest of the notebook consistently filters to this run.

In [ ]:
RUN_ID = str(runs.iloc[0]["run_id"])
if RUN_ID not in set(runs["run_id"]):
    raise ValueError(f"Run is not present in this database: {RUN_ID}")
print(f"Selected run: {RUN_ID}")
display(query("SELECT * FROM run_coverage WHERE run_id=?", [RUN_ID]))

## 4. What arrived, and when?

This is a **receipt-history overview**, not a freshness or synchronization test. `source_time` is the normalized provider time and may be unknown; `received_at_utc` is local receipt/processing time. An active overall feed does not prove that every individual contract updated recently.

In [ ]:
event_summary = query("""
    SELECT kind, count(*) AS received_updates, count(DISTINCT symbol) AS symbols,
           min(received_at_utc) AS first_received_utc,
           max(received_at_utc) AS last_received_utc,
           count(*) FILTER (WHERE source_time IS NULL) AS unknown_source_time,
           count(DISTINCT connection_id) AS connections
    FROM raw_updates WHERE run_id=? GROUP BY kind ORDER BY kind
""", [RUN_ID])
for column in ("first_received_utc", "last_received_utc"):
    event_summary[column.replace("_utc", "_et")] = event_summary[column].dt.tz_convert("America/New_York")
display(event_summary)

## 5. Which instruments were selected?

Selections can be refreshed during a run. Deduplicate their reference rows by streamer symbol before counting contracts. These are the selected pilot instruments, **not all listed SPY options**. Selecting a contract does not by itself prove that it received every event type.

In [ ]:
instruments = query("""
    SELECT symbol, streamer_symbol, option_type, strike, expiration,
           exercise_style, settlement_type, shares_per_contract, stops_trading_at
    FROM instruments WHERE run_id=?
    QUALIFY row_number() OVER (
        PARTITION BY streamer_symbol ORDER BY effective_at DESC, received_at_utc DESC, record_id DESC
    )=1
    ORDER BY expiration NULLS FIRST, option_type, strike
""", [RUN_ID])
print(f"Selected instruments: {len(instruments)} (including the underlying)")
display(instruments.head(20))
display(instruments.loc[instruments["option_type"].notna()].groupby(
    ["expiration", "option_type"], dropna=False
).size().rename("contracts").reset_index())

## 6. Inspect individual quote updates

These are the last ten delivered quote updates in the selected run, not averaged prices and not necessarily ten different symbols. Normalized invalid/non-finite prices may be null; the original representation is retained in the raw payload.

`receive_sequence` orders deliveries within the run. `received_monotonic_ns` measures local timing without wall-clock adjustments; do not compare its epoch across processes. `connection_id` distinguishes reconnects.

In [ ]:
quote_preview = query("""
    SELECT u.receive_sequence, u.connection_id, u.received_at_utc,
           u.received_monotonic_ns, q.symbol,
           q.bid, q.ask, q.bid_size, q.ask_size,
           q.bid_time, q.ask_time, q.source_time, q.record_id AS quote_id
    FROM raw_updates u JOIN quotes q ON q.record_id=u.observation_id
    WHERE u.run_id=? AND u.kind='Quote'
    ORDER BY u.receive_sequence DESC LIMIT 10
""", [RUN_ID])
display(quote_preview)

### Follow one update to its original decoded payload

Change `EVENT_KIND` (`Quote`, `Trade`, `Greeks`, or `Candle`) and `EVENT_SYMBOL` to inspect another example. Candle symbols include their interval/session suffix; choose an exact symbol from the received data when inspecting candles. These payloads contain requested market-data fields, not authentication frames or original WebSocket wire bytes.

In [ ]:
EVENT_KIND = "Quote"
EVENT_SYMBOL = "SPY"
raw_example = query("""
    SELECT record_id, observation_id, kind, symbol, receive_sequence,
           received_at_utc, source_time, raw_record_id, payload
    FROM raw_update_payloads
    WHERE run_id=? AND kind=? AND symbol=?
    ORDER BY receive_sequence LIMIT 1
""", [RUN_ID, EVENT_KIND, EVENT_SYMBOL])
if raw_example.empty:
    print(f"No {EVENT_KIND} update for {EVENT_SYMBOL} in the selected run.")
else:
    display(raw_example.drop(columns="payload"))
    print(json.dumps(json.loads(raw_example.iloc[0]["payload"]), indent=2))

## 7. Compare the minute-sampling layer

A cycle is a scheduled sampling attempt; a sample is an option row from an attempt. `missed` cycles record unavailable earlier intervals rather than fabricated historical quotes. A sampled cycle can still be late or have missing inputs.

The existing sampler requires source-time alignment. Missing provider quote timestamps can leave `underlying_quote_id` and `underlying_mid` empty and `quote_eligible` false, even though SPY updates exist in the raw layer. Receipt-time reconstruction will be a separate, explicitly labeled later analysis.

In [ ]:
display(query("""
    SELECT status, count(*) AS cycles, min(scheduled_at) AS first_scheduled_utc,
           max(scheduled_at) AS last_scheduled_utc, max(lateness_seconds) AS max_lateness_seconds
    FROM cycles WHERE run_id=? GROUP BY status ORDER BY status
""", [RUN_ID]))
sample_preview = query("""
    SELECT scheduled_at, sampled_at, symbol, option_mid, underlying_mid,
           quote_eligible, analytics_fresh, quote_age_seconds,
           analytics_age_seconds, source_skew_seconds,
           quote_id, underlying_quote_id, analytics_id, quality
    FROM samples WHERE run_id=?
    ORDER BY scheduled_at DESC, symbol LIMIT 10
""", [RUN_ID])
display(sample_preview)
if not sample_preview.empty:
    print("Quality flags for the first preview row:")
    display(pd.DataFrame(json.loads(sample_preview.iloc[0]["quality"])))

## 8. Broker analytics and candle examples

Greeks and IV are broker/provider model outputs, not solver ground truth. Their conventions, rates and dividend assumptions are not certified. `analytics.price` is not relabeled as a broker mark.

Candles are provider aggregates, separate from quote events. `latest_candles` chooses a received revision; `elapsed_with_grace` is not provider finality. A later historical candle retrieval does not establish historical point-in-time availability.

In [ ]:
display(query("""
    SELECT symbol, received_at_utc, source_time, price, volatility,
           delta, gamma, theta, vega, rho, event_flags
    FROM analytics WHERE run_id=?
    ORDER BY received_at_utc DESC, record_id LIMIT 5
""", [RUN_ID]))
display(query("""
    SELECT symbol, interval_start, interval_end, received_at_utc,
           open, high, low, close, volume, count,
           elapsed_with_grace, quality
    FROM latest_candles WHERE run_id=?
    ORDER BY interval_start DESC, symbol LIMIT 5
""", [RUN_ID]))

## 9. What operational context was recorded?

These summaries help identify topics for the next investigation. Request success means a request returned or was sent, not that every subscribed symbol delivered data. Diagnostic counts are not automatically counts of bad quotes; some codes describe normal counters or acknowledged limitations.

In [ ]:
display(query("""
    SELECT code, count(*) AS records, min(received_at_utc) AS first_recorded_utc,
           max(received_at_utc) AS last_recorded_utc
    FROM diagnostics WHERE run_id=? GROUP BY code ORDER BY records DESC, code
""", [RUN_ID]))
display(query("""
    SELECT operation, status, count(*) AS requests,
           min(started_at) AS first_started_utc, max(ended_at) AS last_ended_utc
    FROM requests WHERE run_id=? GROUP BY operation, status ORDER BY operation, status
""", [RUN_ID]))

## 10. Quote resolution per minute

### Why investigate receipt-time resolution?

**Observed on October 30, 2026:** the main SPY capture has unknown quote source times, and the small SPY and actively updating Bitcoin/Ethereum quote demos print `event_time=0`, `bid_time=0`, and `ask_time=0`. Zero is treated as an unknown timestamp, not a valid market-event time. The collector still records local UTC receipt times and monotonic receive timing.

**Existing upstream reports:** [tastyware/tastytrade#195: Streaming quotes returning neither ask_time nor bid_time](https://github.com/tastyware/tastytrade/issues/195) describes the same zero bid/ask times. @Graeme22, the SDK maintainer, [responded](https://github.com/tastyware/tastytrade/issues/195#issuecomment-2585349870) that this is what the raw API data looks like and suggested a possible limitation of Tastytrade's partnership with dxFeed. The earlier [tastyware/tastytrade#119: Missing bidTime and askTime in EventType.QUOTE](https://github.com/tastyware/tastytrade/issues/119) also reproduced missing DXLink timestamps after an SDK upgrade. Both issues are closed, but neither documents a demonstrated fix for the missing bid/ask times.

**Working hypothesis, not confirmed cause:** timestamp availability is probably an upstream feed/service limitation related to the dxFeed-Tastytrade broker agreement, rather than a bug in our collector. The maintainer's suggestion about licensing/payment is speculation; we have not confirmed the agreement's terms or the cause with Tastytrade support. The reports directly address bid/ask times and do not establish the semantics of every other zero field. We are not attempting to bypass feed restrictions.

**Implication for BSM validation:** without source bid/ask times, we cannot prove exchange-time freshness, transport latency, or option/SPY synchronization. We will investigate whether frequent local receipts and sufficiently stable prices over short intervals can support explicitly approximate, receipt-time-based comparisons. This section starts with quote counts per minute; within-minute arrival gaps, bid/ask/midpoint variability, and option/SPY pairing remain separate checks. Frequent updates alone do not certify the data or replace the missing source timestamps.

### Counting method

Count **received Quote deliveries**, including repeated payloads and invalid prices, not distinct prices or exchange ticks. Minute buckets use local UTC receipt time, with New York labels for inspection. Only the selected run and its SPY options are included; SPY itself is excluded.

Build every minute from each option's first recorded selection through the earlier of run end, regular-session close, or contract trading stop. Selection time is a proxy for intended subscription exposure, not proof that subscription was active. Setup delays and reconnect gaps remain in the denominator; minutes before selection do not. A zero-update minute may mean an unchanged market or a delivery gap.

Statistics use **complete 60-second minutes only**, including zeros. Partial boundary minutes and their deliveries are retained separately, not scaled up. Schema-1 runs cannot support this analysis. Counts alone do not establish uniform arrival spacing, low price variance, source freshness, or BSM suitability; those are later checks.

In [ ]:
recording_mode = query("SELECT json_extract_string(capabilities, '$.recording_mode') AS mode FROM runs WHERE run_id=?", [RUN_ID])
if recording_mode['mode'].tolist() != ['all_received_updates_and_minute_samples']:
    raise ValueError('Select a run that retained all received updates; minute-only runs cannot measure delivery frequency.')

quote_minute_counts = query("""
    WITH options AS (
        SELECT streamer_symbol, min(effective_at) AS first_selected_at,
               min(stops_trading_at) AS stops_trading_at,
               min(expiration) AS expiration, min(option_type) AS option_type,
               min(strike) AS strike
        FROM instruments WHERE run_id=? AND option_type IS NOT NULL
          AND streamer_symbol LIKE '.SPY%'
        GROUP BY streamer_symbol
    ), bounds AS (
        SELECT o.*, greatest(r.started_at, r.session_open, o.first_selected_at) AS exposure_start,
               least(r.ended_at, r.session_close, o.stops_trading_at) AS exposure_end
        FROM options o CROSS JOIN runs r WHERE r.run_id=? AND r.ended_at IS NOT NULL
    ), grid AS (
        SELECT b.*, m.minute_utc,
               greatest(b.exposure_start, m.minute_utc) AS interval_start,
               least(b.exposure_end, m.minute_utc + INTERVAL '1 minute') AS interval_end
        FROM bounds b CROSS JOIN LATERAL generate_series(
            date_trunc('minute', b.exposure_start), date_trunc('minute', b.exposure_end),
            INTERVAL '1 minute'
        ) m(minute_utc)
        WHERE b.exposure_end > b.exposure_start AND m.minute_utc < b.exposure_end
    )
    SELECT g.streamer_symbol, g.expiration, g.option_type, g.strike,
           g.first_selected_at, g.minute_utc,
           epoch(g.interval_end - g.interval_start) AS exposure_seconds,
           g.interval_end - g.interval_start = INTERVAL '1 minute' AS full_minute,
           count(u.record_id) AS quote_count
    FROM grid g LEFT JOIN raw_updates u
      ON u.run_id=? AND u.kind='Quote' AND u.symbol=g.streamer_symbol
     AND u.received_at_utc >= g.interval_start AND u.received_at_utc < g.interval_end
    GROUP BY ALL ORDER BY g.streamer_symbol, g.minute_utc
""", [RUN_ID, RUN_ID, RUN_ID])
if quote_minute_counts.empty:
    raise ValueError('No selected SPY-option exposure intervals found in this completed run.')
quote_minute_counts['minute_et'] = quote_minute_counts['minute_utc'].dt.tz_convert('America/New_York')
full_quote_minutes = quote_minute_counts.loc[quote_minute_counts['full_minute']].copy()

option_quote_count_stats = full_quote_minutes.groupby('streamer_symbol')['quote_count'].agg(
    full_minutes='size', full_minute_quotes='sum', zero_quote_minutes=lambda x: int(x.eq(0).sum()),
    min_per_minute='min', p05_per_minute=lambda x: x.quantile(0.05),
    median_per_minute='median', mean_per_minute='mean', std_per_minute='std',
    p95_per_minute=lambda x: x.quantile(0.95), max_per_minute='max',
).reset_index()
exposure_summary = quote_minute_counts.groupby('streamer_symbol').agg(
    expiration=('expiration', 'first'), option_type=('option_type', 'first'), strike=('strike', 'first'),
    first_selected_at=('first_selected_at', 'first'), exposed_minutes=('quote_count', 'size'),
    total_quotes=('quote_count', 'sum'),
).reset_index()
partial_summary = quote_minute_counts.loc[~quote_minute_counts['full_minute']].groupby('streamer_symbol').agg(
    partial_minutes=('quote_count', 'size'), partial_minute_quotes=('quote_count', 'sum'),
).reset_index()
option_quote_count_stats = exposure_summary.merge(option_quote_count_stats, on='streamer_symbol', how='left').merge(
    partial_summary, on='streamer_symbol', how='left'
)
count_columns = ['full_minutes', 'full_minute_quotes', 'zero_quote_minutes', 'partial_minutes', 'partial_minute_quotes']
option_quote_count_stats[count_columns] = option_quote_count_stats[count_columns].fillna(0).astype('int64')
denominator = option_quote_count_stats['full_minutes'].replace(0, float('nan'))
option_quote_count_stats['zero_quote_pct'] = 100 * option_quote_count_stats['zero_quote_minutes'] / denominator
option_quote_count_stats = option_quote_count_stats.sort_values(['expiration', 'option_type', 'strike']).reset_index(drop=True)
assert (option_quote_count_stats['total_quotes'] == option_quote_count_stats['full_minute_quotes'] + option_quote_count_stats['partial_minute_quotes']).all()
print(f"Options: {len(option_quote_count_stats)}; option-minute buckets: {len(quote_minute_counts):,}")
print(f"Received option quotes in these exposure intervals: {quote_minute_counts['quote_count'].sum():,}")
print('std_per_minute is sample standard deviation of counts, not price variance; undefined with fewer than two full minutes.')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    formatted_stats = option_quote_count_stats.copy()
    numeric_columns = formatted_stats.select_dtypes(include='number').columns
    formatted_stats[numeric_columns] = formatted_stats[numeric_columns].round(2)
    display(formatted_stats)


In [ ]:
# Change this symbol to inspect its minute-by-minute counts, including zeros.
OPTION_TO_INSPECT = str(option_quote_count_stats.iloc[0]['streamer_symbol'])
if OPTION_TO_INSPECT not in set(option_quote_count_stats['streamer_symbol']):
    raise ValueError(f'Option is not present in the selected run: {OPTION_TO_INSPECT}')
option_minute_detail = quote_minute_counts.loc[
    quote_minute_counts['streamer_symbol'].eq(OPTION_TO_INSPECT),
    ['minute_et', 'quote_count', 'exposure_seconds', 'full_minute'],
]
print(f'{OPTION_TO_INSPECT}: first 20 minute buckets; full data remain in option_minute_detail')
display(option_minute_detail.head(20))
print('Distribution of counts across all complete option-minute buckets (not an equally weighted contract average):')
display(full_quote_minutes['quote_count'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame('quotes_per_option_minute'))

### 10.1. Retrospective minute-boundary candidates

We look for **multiple usable option and SPY receipts on both sides of a minute boundary**, small gaps, and stable bid/ask/midpoint prices. This is evidence of local stability, **not proof of exchange-time alignment or absence of feed delay**. Flat or repeated quotes can be stale; distinct payload and receipt-time counts help distinguish repetition from new observations.

The next cell examines symmetric **±5-second receipt windows** by default. Windows must fit entirely inside the run/session/contract exposure interval. The session-close boundary is excluded because its after-side is outside regular hours. Recorded selection remains an exposure proxy. Usable prices require finite bid/ask, bid >= 0, ask > 0, and ask >= bid; sizes are not an eligibility gate. Invalid deliveries are counted but excluded from price calculations.

Option deliveries are matched **backward in receive-sequence order** to the most recent usable SPY quote on the same connection. Pairs must be inside the window and within the configurable receipt-age limit. No later SPY receipt is used, including same-timestamp deliveries received later in sequence. Reused SPY observations are counted explicitly. Pair-age and gap metrics describe local clocks only.

Correlation uses **successive paired midpoint changes**, not price levels, and is shown only with enough changes and distinct SPY observations. It can be undefined in a stable window, noisy in a short window, or share batching effects. Put correlation often has the opposite sign to call correlation. Correlation is **not** a screening requirement or proof of synchronization.

The editable screening defaults are illustrative exploration thresholds, **not approved BSM tolerances**. The screen requires before/after support, close boundary brackets, small window-wide gaps, valid prices, a single shared connection, and low bid/ask/midpoint ranges for both instruments. It does not test absolute spread tightness, source freshness, candle alignment, or broker analytics alignment. All windows and failures remain available. Quiet-window selection introduces selection bias: report the retained fraction and later compare rejected windows.

The before-boundary option observation and its matched SPY observation are retained by ID for follow-up. After-boundary receipts assess stability retrospectively; this look-ahead must not be represented as a live trading/backtest signal. Candle-open/close comparison is a later step, not implied by this quote-only screen.


In [ ]:
import math

# Illustrative exploration controls; edit and rerun, not BSM acceptance criteria.
BOUNDARY_WINDOW_SECONDS = 5.0
MAX_PAIR_AGE_SECONDS = 2.0
MIN_QUOTES_EACH_SIDE = 2
MAX_BOUNDARY_GAP_SECONDS = 2.0
MAX_WINDOW_GAP_SECONDS = 2.0
MAX_OPTION_PRICE_RANGE = 0.02  # Dollars per share, for bid, ask and midpoint.
MAX_SPY_PRICE_RANGE = 0.05
MIN_CORRELATION_CHANGES = 5

positive_controls = [BOUNDARY_WINDOW_SECONDS, MAX_PAIR_AGE_SECONDS,
                     MAX_BOUNDARY_GAP_SECONDS, MAX_WINDOW_GAP_SECONDS,
                     MAX_OPTION_PRICE_RANGE, MAX_SPY_PRICE_RANGE]
if not all(math.isfinite(x) and x > 0 for x in positive_controls):
    raise ValueError('Window, age, gap and price-range controls must be finite and positive.')
if BOUNDARY_WINDOW_SECONDS >= 30:
    raise ValueError('Use a window under 30 seconds so adjacent minute windows do not overlap.')
if any(isinstance(x, bool) or not isinstance(x, int) or x < 1
       for x in [MIN_QUOTES_EACH_SIDE, MIN_CORRELATION_CHANGES]):
    raise ValueError('Count controls must be positive integers.')
mode = query("SELECT json_extract_string(capabilities, '$.recording_mode') AS mode FROM runs WHERE run_id=?", [RUN_ID])
if mode['mode'].tolist() != ['all_received_updates_and_minute_samples']:
    raise ValueError('Boundary analysis requires a run recording all received updates.')

boundary_sql = """
WITH settings AS (
    SELECT ?::VARCHAR AS run_id, ?::DOUBLE AS window_s, ?::DOUBLE AS pair_age_s
), options AS (
    SELECT i.streamer_symbol, min(i.effective_at) AS selected_at,
           min(i.stops_trading_at) AS stops_at, min(i.option_type) AS option_type,
           min(i.expiration) AS expiration, min(i.strike) AS strike
    FROM instruments i JOIN settings s ON i.run_id=s.run_id
    WHERE i.option_type IS NOT NULL AND i.streamer_symbol LIKE '.SPY%'
    GROUP BY i.streamer_symbol
), exposure AS (
    SELECT o.*, greatest(o.selected_at, r.started_at, r.session_open) AS start_at,
           least(r.ended_at, r.session_close, o.stops_at) AS end_at
    FROM options o CROSS JOIN settings s JOIN runs r ON r.run_id=s.run_id
    WHERE r.ended_at IS NOT NULL
), boundaries AS (
    SELECT e.*, t.boundary_utc,
           t.boundary_utc - s.window_s * INTERVAL '1 second' AS window_start,
           t.boundary_utc + s.window_s * INTERVAL '1 second' AS window_end
    FROM exposure e CROSS JOIN settings s CROSS JOIN LATERAL generate_series(
        date_trunc('minute', e.start_at), date_trunc('minute', e.end_at),
        INTERVAL '1 minute') t(boundary_utc)
    WHERE t.boundary_utc - s.window_s * INTERVAL '1 second' >= e.start_at
      AND t.boundary_utc + s.window_s * INTERVAL '1 second' <= e.end_at
), windows AS (
    SELECT streamer_symbol AS symbol, boundary_utc, window_start, window_end FROM boundaries
    UNION
    SELECT 'SPY', boundary_utc, window_start, window_end FROM boundaries
), quotes_received AS (
    SELECT u.record_id AS delivery_id, u.observation_id, u.raw_record_id,
           u.symbol, u.received_at_utc AS receipt, u.receive_sequence AS seq,
           u.connection_id, q.bid, q.ask, (q.bid+q.ask)/2 AS mid,
           coalesce(isfinite(q.bid) AND isfinite(q.ask)
               AND q.bid >= 0 AND q.ask > 0 AND q.ask >= q.bid, false) AS valid
    FROM raw_updates u JOIN settings s ON u.run_id=s.run_id
    LEFT JOIN quotes q ON q.record_id=u.observation_id AND q.run_id=u.run_id
    WHERE u.kind='Quote'
), window_rows AS (
    SELECT w.*, q.*, lag(q.receipt) OVER (
        PARTITION BY w.symbol, w.boundary_utc ORDER BY q.receipt, q.seq
    ) AS previous_receipt
    FROM windows w JOIN quotes_received q ON q.symbol=w.symbol
      AND q.receipt >= w.window_start AND q.receipt < w.window_end
    WHERE q.valid
), delivery_counts AS (
    SELECT w.symbol, w.boundary_utc, count(q.delivery_id) AS deliveries,
           count(q.delivery_id) FILTER (WHERE NOT q.valid) AS invalid_deliveries
    FROM windows w LEFT JOIN quotes_received q ON q.symbol=w.symbol
      AND q.receipt >= w.window_start AND q.receipt < w.window_end
    GROUP BY w.symbol, w.boundary_utc
), price_stats AS (
    SELECT symbol, boundary_utc, count(*) AS valid_quotes,
           count(*) FILTER (WHERE receipt < boundary_utc) AS before_quotes,
           count(*) FILTER (WHERE receipt >= boundary_utc) AS after_quotes,
           count(DISTINCT receipt) AS distinct_receipts,
           count(DISTINCT receipt) FILTER (WHERE receipt<boundary_utc) AS before_distinct_receipts,
           count(DISTINCT receipt) FILTER (WHERE receipt>=boundary_utc) AS after_distinct_receipts,
           count(DISTINCT raw_record_id) AS distinct_payloads,
           count(DISTINCT connection_id) AS connections,
           min(connection_id) AS connection_id,
           min(mid) AS mid_min, max(mid) AS mid_max,
           max(mid)-min(mid) AS mid_range, stddev_samp(mid) AS mid_std,
           max(bid)-min(bid) AS bid_range, max(ask)-min(ask) AS ask_range,
           median(ask-bid) AS median_spread,
           epoch(boundary_utc-max(receipt) FILTER (WHERE receipt < boundary_utc)) AS before_gap_s,
           epoch(min(receipt) FILTER (WHERE receipt >= boundary_utc)-boundary_utc) AS after_gap_s,
           greatest(epoch(min(receipt)-min(window_start)),
                    epoch(min(window_end)-max(receipt)),
                    coalesce(max(epoch(receipt-previous_receipt)), 0)) AS max_window_gap_s
    FROM window_rows GROUP BY symbol, boundary_utc
), matched AS (
    SELECT oq.*, sq.observation_id AS spy_observation_id, sq.connection_id AS spy_connection_id,
           sq.receipt AS spy_receipt, sq.mid AS spy_mid, sq.seq AS spy_seq,
           epoch(oq.receipt-sq.receipt) AS pair_age_s
    FROM (SELECT * FROM quotes_received WHERE valid AND symbol<>'SPY') oq
    ASOF LEFT JOIN (SELECT * FROM quotes_received WHERE valid AND symbol='SPY') sq
      ON oq.connection_id=sq.connection_id AND oq.seq>=sq.seq
), window_pairs AS (
    SELECT b.streamer_symbol, b.boundary_utc, m.*
    FROM boundaries b JOIN matched m ON m.symbol=b.streamer_symbol
      AND m.receipt>=b.window_start AND m.receipt<b.window_end
    CROSS JOIN settings s
    WHERE m.spy_receipt>=b.window_start AND m.spy_receipt<b.window_end
      AND m.pair_age_s>=0 AND m.pair_age_s<=s.pair_age_s
), changes AS (
    SELECT *, mid-lag(mid) OVER w AS option_change,
           spy_mid-lag(spy_mid) OVER w AS spy_change
    FROM window_pairs
    WINDOW w AS (PARTITION BY streamer_symbol, boundary_utc, connection_id ORDER BY seq)
), pair_stats AS (
    SELECT streamer_symbol, boundary_utc, count(*) AS paired_quotes,
           count(*) FILTER (WHERE receipt<boundary_utc) AS before_pairs,
           count(*) FILTER (WHERE receipt>=boundary_utc) AS after_pairs,
           count(DISTINCT spy_observation_id) AS distinct_paired_spy,
           max(pair_age_s) AS max_pair_age_s, median(pair_age_s) AS median_pair_age_s,
           count(option_change) AS change_pairs,
           corr(option_change, spy_change) AS change_correlation,
           arg_max(observation_id, seq) FILTER (WHERE receipt<boundary_utc) AS option_observation_id,
           arg_max(spy_observation_id, seq) FILTER (WHERE receipt<boundary_utc) AS spy_observation_id,
           arg_max(mid, seq) FILTER (WHERE receipt<boundary_utc) AS option_mid_before,
           arg_max(spy_mid, seq) FILTER (WHERE receipt<boundary_utc) AS spy_mid_before,
           arg_max(receipt, seq) FILTER (WHERE receipt<boundary_utc) AS option_receipt_before,
           arg_max(spy_receipt, seq) FILTER (WHERE receipt<boundary_utc) AS spy_receipt_before
    FROM changes GROUP BY streamer_symbol, boundary_utc
)
SELECT b.streamer_symbol, b.option_type, b.expiration, b.strike, b.boundary_utc,
       dc.deliveries, dc.invalid_deliveries,
       ps.* EXCLUDE (symbol, boundary_utc),
       p.* EXCLUDE (streamer_symbol, boundary_utc)
FROM boundaries b JOIN delivery_counts dc ON dc.symbol=b.streamer_symbol AND dc.boundary_utc=b.boundary_utc
LEFT JOIN price_stats ps ON ps.symbol=b.streamer_symbol AND ps.boundary_utc=b.boundary_utc
LEFT JOIN pair_stats p ON p.streamer_symbol=b.streamer_symbol AND p.boundary_utc=b.boundary_utc
ORDER BY b.streamer_symbol, b.boundary_utc
"""
boundary_option_stats = query(boundary_sql, [RUN_ID, BOUNDARY_WINDOW_SECONDS, MAX_PAIR_AGE_SECONDS])
if boundary_option_stats.empty:
    raise ValueError('No complete symmetric windows fit this run and option exposure.')
# Calculate SPY window statistics once per boundary using the same SQL CTEs.
spy_sql = boundary_sql[:boundary_sql.rindex('\nSELECT b.streamer_symbol')] + """
SELECT dc.boundary_utc, dc.deliveries AS spy_deliveries,
       dc.invalid_deliveries AS spy_invalid_deliveries,
       ps.* EXCLUDE (symbol, boundary_utc)
FROM delivery_counts dc LEFT JOIN price_stats ps
  ON ps.symbol=dc.symbol AND ps.boundary_utc=dc.boundary_utc
WHERE dc.symbol='SPY'
"""
spy_stats = query(spy_sql, [RUN_ID, BOUNDARY_WINDOW_SECONDS, MAX_PAIR_AGE_SECONDS])
spy_stats = spy_stats.rename(columns={c: 'spy_'+c for c in spy_stats.columns
                                     if c != 'boundary_utc' and not c.startswith('spy_')})
boundary_candidates = boundary_option_stats.merge(spy_stats, on='boundary_utc', how='left', validate='many_to_one')
count_fields = ['valid_quotes', 'before_quotes', 'after_quotes', 'distinct_receipts',
                'before_distinct_receipts', 'after_distinct_receipts',
                'distinct_payloads', 'connections', 'paired_quotes', 'before_pairs',
                'after_pairs', 'distinct_paired_spy', 'change_pairs']
for prefix in ['', 'spy_']:
    for field in count_fields:
        column = prefix+field
        if column in boundary_candidates:
            boundary_candidates[column] = boundary_candidates[column].fillna(0).astype('int64')
b = boundary_candidates
adequate_correlation = (b.change_pairs >= MIN_CORRELATION_CHANGES) & (b.distinct_paired_spy >= 3)
b['change_correlation'] = b.change_correlation.where(adequate_correlation & b.change_correlation.map(math.isfinite))
b['direction_adjusted_correlation'] = b.change_correlation * b.option_type.map({'C': 1, 'P': -1})
b['boundary_et'] = b.boundary_utc.dt.tz_convert('America/New_York')
b['paired_quote_pct'] = 100*b.paired_quotes / b.valid_quotes.replace(0, float('nan'))
checks = {
    'valid_prices': b.invalid_deliveries.eq(0) & b.spy_invalid_deliveries.eq(0),
    'both_sides': (b.before_quotes>=MIN_QUOTES_EACH_SIDE) & (b.after_quotes>=MIN_QUOTES_EACH_SIDE)
                 & (b.spy_before_quotes>=MIN_QUOTES_EACH_SIDE) & (b.spy_after_quotes>=MIN_QUOTES_EACH_SIDE),
    'distinct_receipts': (b.before_distinct_receipts>=MIN_QUOTES_EACH_SIDE)
                         & (b.after_distinct_receipts>=MIN_QUOTES_EACH_SIDE)
                         & (b.spy_before_distinct_receipts>=MIN_QUOTES_EACH_SIDE)
                         & (b.spy_after_distinct_receipts>=MIN_QUOTES_EACH_SIDE),
    'one_shared_connection': b.connections.eq(1) & b.spy_connections.eq(1) & b.connection_id.eq(b.spy_connection_id),
    'boundary_brackets': b[['before_gap_s','after_gap_s','spy_before_gap_s','spy_after_gap_s']].le(MAX_BOUNDARY_GAP_SECONDS).all(axis=1),
    'window_gaps': b[['max_window_gap_s','spy_max_window_gap_s']].le(MAX_WINDOW_GAP_SECONDS).all(axis=1),
    'option_stability': b[['mid_range','bid_range','ask_range']].le(MAX_OPTION_PRICE_RANGE+1e-12).all(axis=1),
    'spy_stability': b[['spy_mid_range','spy_bid_range','spy_ask_range']].le(MAX_SPY_PRICE_RANGE+1e-12).all(axis=1),
    'backward_pairs': b.before_pairs.ge(MIN_QUOTES_EACH_SIDE) & b.after_pairs.ge(MIN_QUOTES_EACH_SIDE),
}
boundary_checks = pd.DataFrame(checks, index=b.index).fillna(False)
b['exploratory_candidate'] = boundary_checks.all(axis=1)
b['failed_checks'] = boundary_checks.apply(lambda row: ', '.join(row.index[~row]), axis=1)
boundary_candidates = b
quiet_boundary_candidates = b.loc[b.exploratory_candidate].copy()
print(f'±{BOUNDARY_WINDOW_SECONDS:g}s; backward SPY receipt-age limit {MAX_PAIR_AGE_SECONDS:g}s.')
print(f'{len(quiet_boundary_candidates):,} / {len(b):,} option-boundary windows pass illustrative screen ({100*b.exploratory_candidate.mean():.2f}%).')
print('No claim of source-time synchronization or BSM eligibility. Correlation is diagnostic only.')
display(pd.DataFrame({'check': boundary_checks.columns,
                      'passing_windows': boundary_checks.sum().values,
                      'passing_pct': 100*boundary_checks.mean().values}))
boundary_contract_summary = b.groupby('streamer_symbol').agg(
    windows=('exploratory_candidate','size'), candidates=('exploratory_candidate','sum'),
    median_option_range=('mid_range','median'), median_spy_range=('spy_mid_range','median'),
    median_pair_age_s=('median_pair_age_s','median'),
).reset_index()
boundary_contract_summary['candidate_pct'] = 100*boundary_contract_summary.candidates/boundary_contract_summary.windows
with pd.option_context('display.max_rows', None):
    display(boundary_contract_summary)


In [ ]:
# Inspect candidates and rejected windows for the same contract, not just winners.
BOUNDARY_OPTION_TO_INSPECT = OPTION_TO_INSPECT
if BOUNDARY_OPTION_TO_INSPECT not in set(boundary_candidates.streamer_symbol):
    raise ValueError('Choose an option present in boundary_candidates.')
boundary_columns = [
    'streamer_symbol', 'boundary_et', 'exploratory_candidate', 'failed_checks',
    'before_quotes', 'after_quotes', 'spy_before_quotes', 'spy_after_quotes',
    'distinct_payloads', 'spy_distinct_payloads', 'before_gap_s', 'after_gap_s',
    'spy_before_gap_s', 'spy_after_gap_s', 'max_window_gap_s', 'spy_max_window_gap_s',
    'mid_range', 'bid_range', 'ask_range', 'median_spread',
    'spy_mid_range', 'spy_bid_range', 'spy_ask_range', 'spy_median_spread',
    'paired_quotes', 'distinct_paired_spy', 'paired_quote_pct', 'max_pair_age_s',
    'change_pairs', 'change_correlation', 'direction_adjusted_correlation',
    'option_mid_before', 'spy_mid_before', 'option_receipt_before', 'spy_receipt_before',
    'option_observation_id', 'spy_observation_id',
]
print('First 20 candidate windows; complete set: quiet_boundary_candidates')
display(quiet_boundary_candidates[boundary_columns].head(20))
print(f'{BOUNDARY_OPTION_TO_INSPECT}: first 20 windows, including failures')
boundary_option_detail = boundary_candidates.loc[
    boundary_candidates.streamer_symbol.eq(BOUNDARY_OPTION_TO_INSPECT), boundary_columns
]
display(boundary_option_detail.head(20))
print('Compare all/retained/rejected distributions to expose quiet-window selection bias:')
comparison_metrics = ['mid_range', 'spy_mid_range', 'median_spread', 'spy_median_spread',
                      'max_window_gap_s', 'spy_max_window_gap_s', 'max_pair_age_s',
                      'change_correlation']
for label, frame in [('all', boundary_candidates),
                     ('retained', quiet_boundary_candidates),
                     ('rejected', boundary_candidates.loc[~boundary_candidates.exploratory_candidate])]:
    print(f'{label}: {len(frame):,} windows')
    display(frame[comparison_metrics].describe(percentiles=[0.05, 0.5, 0.95]))


## 11. Preliminary BSM validation: limitations, compromises, and handoff

### 11.1. What this capture can establish

Our goal is **preliminary, uncertainty-aware validation**, not certification of market-time synchronization. The intended BSM use operates at timescales longer than one minute. Sub-minute exploration helps characterize input uncertainty; it is not a requirement to demonstrate tick-level accuracy. Slower intended use does not eliminate timing risk during rapid moves, near expiration, or for sensitive/low-vega contracts.

The October 30, 2026 main capture covers only part of one regular session and a selected pilot universe, not every SPY option. Raw receipt history begins after the open. A short sample can reveal gross formula/input/convention problems, but cannot establish reliability across market regimes, dividend windows, or historical periods. Repeated minutes and contracts sharing the same SPY observations are not independent tests.

### 11.2. Known issues and explicit compromises

| Issue | Compromise and reporting rule |
| --- | --- |
| SPY and option Quote source bid/ask times are unknown | Use recorded local receipts and receive sequence for explicitly approximate backward pairing. Keep source times null with reasons; never fabricate them or relabel receipt time as source time. Transport delay, source freshness and market-time synchronization remain unknown. |
| Upstream reports reproduce zero bid/ask times | [tastyware/tastytrade#195](https://github.com/tastyware/tastytrade/issues/195) and [tastyware/tastytrade#119](https://github.com/tastyware/tastytrade/issues/119) support an upstream limitation. The maintainer's [dxFeed/broker partnership explanation](https://github.com/tastyware/tastytrade/issues/195#issuecomment-2585349870) is a working hypothesis, not confirmed licensing terms or a support response. |
| Correlation cannot prove synchronization | Do not make correlation an acceptance gate. Flat series may have undefined correlation, and correlated series can still be delayed or batched. Retain correlation only as an exploratory diagnostic. |
| Quotes near a boundary may describe slightly different market instants | Prefer locally stable windows for a preliminary subset; perturb option/underlying inputs over observed ranges and plausible pairing offsets. Report sensitivity, not a fabricated timing confidence interval. Stability does not rule out stale or delayed data. |
| Quiet-window selection biases the sample | Preserve all examined windows, flags, thresholds, and rejection reasons alongside any selected subset. Report counts of distinct sessions, boundaries, contracts, expirations and coverage, not just row counts. The Section 10.1 thresholds are illustrative, not approved BSM tolerances. |
| Retrospective screening uses receipts after the boundary | Keep that look-ahead explicit. The selected option/SPY price observations are backward receipts, but their after-side stability assessment was not available at the boundary. Do not use this subset as a point-in-time trading/backtest signal. |
| Candle opens/closes are not contemporaneous quotes | Candle/trade/Greeks source-time fields exist with different semantics. Keep candle interval labels, revisions, source and receipt times separate. Candle-open/close substitution is a later sensitivity comparison, not proof of quote alignment. The symmetric screen excludes the regular-session close boundary. |
| Broker IV/Greeks are model outputs with unverified conventions | Retain them as references with independent timing and provenance. Unknown units, model, rate/dividend inputs and analytics freshness prevent a certified numerical comparison. Do not call analytics.price a broker mark. |
| BSM needs additional model inputs and conventions | Macro Ideas must specify rate/dividend assumptions, valuation-time/expiry conventions, and price/Greek units. SPY options are American-style while standard BSM is European; exercise/dividend effects can cause genuine market-model discrepancies. No values or conventions are silently supplied by this export. |
| Price agreement alone is not independent formula validation | Compare bid/mid/ask scenarios and input sensitivities; bid/ask ranges are not confidence intervals or guaranteed fills. Inferring IV from a price and repricing that same price is a round-trip consistency test, not independent market validation. Retain failed solves, no-arbitrage violations and low-vega cases in downstream reports. |

Existing collector `quote_eligible` and `broker_comparison_ready` flags are **not overridden** by an exploratory quiet-window label. No synchronization, numerical BSM validation, handoff acceptance, historical-input approval, or strategy replay has been demonstrated here. Independent known-input/QuantLib-style reference tests remain necessary in Macro Ideas.

### 11.3. First validation dataset: proposed handoff contract

The first extract is a **versioned research fixture**, not a claim that the original strict requirements passed. Keep immutable capture rows separate from derived receipt-time pairs and exploratory selection flags. We chose to retain all examined boundary windows with the quiet-candidate flag, using the shared DuckDB as the source dependency. The preparation and opt-in writing cells below implement this handoff.

- Carry the selected run ID, session date, recording mode, collector/schema/SDK versions, capture coverage, and the actual boundary-window/pairing/screening settings in a manifest. Include this limitations statement with the exported bundle so it survives transfer.
- Keep contract identifiers, option type, strike, expiration/expires_at/stops_trading_at, exercise style and available deliverable terms. Preserve unknowns explicitly.
- Include evaluation boundary time **as a research clock**, option/underlying receipt times, observation/delivery/raw IDs, connection and receive sequence, bid/ask/sizes, separately computed midpoints, local pairing age and window stability statistics. Keep all source-time fields and quality flags unchanged.
- Preserve linked broker analytics and option/SPY candles as separately timestamped observations, not silently synchronized inputs. Define any backward analytics match and candle interval/revision choice before export; unmatched references stay null with reasons.
- Retain sanitized raw payloads and reference provenance sufficient to resolve exported IDs. Export portable Parquet tables plus a machine-readable manifest/schema, counts and checksums; keep outputs under the Git-ignored exports directory. Do not include authentication/account payloads.
- Do not infer rates, dividend yield, broker mark, model eligibility or successful solver outputs in this repository. Those are explicit downstream decisions. The bundle supplies data and uncertainty evidence; BSM calculations continue in Macro Ideas.

The [Macro Ideas collector handoff](https://github.com/khazaddim/Macro_Ideas/blob/main/short_put_replay/quote_collector_requirements.md) keeps collection, numerical comparison and historical approval separate. Its strict source-time freshness/alignment criteria remain unresolved; this preliminary fixture is an explicitly documented compromise, not a replacement or silent relaxation of that contract.

### 11.4. Version 1: shared-database validation fixture

Macro Ideas can read the same capture DuckDB on this computer. We therefore export **all examined boundary windows**, retaining `exploratory_candidate` and rejection reasons, rather than copying the full capture. The candidate flag is an exploratory stability screen, **not source-time or numerical BSM eligibility**.

The derived Parquet contains the before-boundary paired option/SPY quotes when available, their prices/sizes/source times and original IDs, first-recorded contract terms/provenance, and all window diagnostics. Missing pairs remain rows with null quote fields. Contract terms use the first recorded reference for each symbol in this run; later reference revisions remain in DuckDB for downstream review. Broker analytics and candles are **not silently attached**: Macro Ideas must choose and document their independent alignment policies using the shared database.

This is **not a self-contained export of the original collector contract**. The source database is a required dependency for raw payloads, analytics, candles, revision history and original strict minute-sample flags. The manifest includes its path, checksum, run metadata, selection settings, and the Section 11 limitations. Preserve that database or update its location in the downstream loader. Never treat missing source times as fresh or set unknown model inputs to defaults.

The writing cell is opt-in and refuses to overwrite an existing export directory. To regenerate, use a new version/directory. The clean source defaults to no write; the private executed copy records the explicitly requested export. Source capture tables are never changed.


In [ ]:
import hashlib

VALIDATION_SCHEMA_VERSION = 1
validation_dataset = boundary_candidates.copy()
if validation_dataset.duplicated(['streamer_symbol', 'boundary_utc']).any():
    raise ValueError('Duplicate option-boundary keys; export refused.')
pair_ids = sorted(set(validation_dataset.option_observation_id.dropna())
                  | set(validation_dataset.spy_observation_id.dropna()))
quote_references = query("""
    SELECT q.record_id AS observation_id, q.raw_record_id,
           q.received_at_utc, q.source_time, q.bid_time, q.ask_time,
           q.bid, q.ask, q.bid_size, q.ask_size,
           u.record_id AS delivery_id, u.receive_sequence, u.received_monotonic_ns,
           u.connection_id
    FROM quotes q JOIN raw_updates u ON u.observation_id=q.record_id
      AND u.run_id=q.run_id AND u.kind='Quote'
    WHERE q.run_id=? AND q.record_id IN (SELECT unnest(?::VARCHAR[]))
""", [RUN_ID, pair_ids])
if quote_references.observation_id.duplicated().any():
    raise ValueError('Ambiguous observation-to-delivery mapping; export refused.')
if set(quote_references.observation_id) != set(pair_ids):
    raise ValueError('Some exported quote observation IDs cannot be resolved.')
for prefix in ['option', 'spy']:
    references = quote_references.rename(columns={c: prefix+'_'+c for c in quote_references.columns})
    validation_dataset = validation_dataset.merge(
        references, on=prefix+'_observation_id', how='left', validate='many_to_one',
        suffixes=('', '_reference'),
    )
contract_references = query("""
    SELECT streamer_symbol, record_id AS contract_reference_id,
           raw_record_id AS contract_raw_record_id, effective_at AS contract_reference_at,
           symbol AS contract_symbol, expires_at, stops_trading_at,
           exercise_style, settlement_type, shares_per_contract, terms_status
    FROM instruments WHERE run_id=? AND option_type IS NOT NULL
      AND streamer_symbol LIKE '.SPY%'
    QUALIFY row_number() OVER (
        PARTITION BY streamer_symbol ORDER BY effective_at, received_at_utc, record_id
    )=1
""", [RUN_ID])
validation_dataset = validation_dataset.merge(contract_references, on='streamer_symbol',
                                               how='left', validate='many_to_one')
if validation_dataset.contract_reference_id.isna().any():
    raise ValueError('Missing contract reference for an exported window.')
validation_settings = {
    'window_seconds_each_side': BOUNDARY_WINDOW_SECONDS,
    'max_pair_receipt_age_seconds': MAX_PAIR_AGE_SECONDS,
    'min_quotes_each_side': MIN_QUOTES_EACH_SIDE,
    'max_boundary_gap_seconds': MAX_BOUNDARY_GAP_SECONDS,
    'max_window_gap_seconds': MAX_WINDOW_GAP_SECONDS,
    'max_option_bid_ask_mid_range_dollars': MAX_OPTION_PRICE_RANGE,
    'max_spy_bid_ask_mid_range_dollars': MAX_SPY_PRICE_RANGE,
    'min_correlation_changes': MIN_CORRELATION_CHANGES,
    'correlation_is_selection_gate': False,
}
settings_fingerprint = hashlib.sha256(json.dumps(validation_settings, sort_keys=True).encode()).hexdigest()
validation_dataset['run_id'] = RUN_ID
validation_dataset['validation_schema_version'] = VALIDATION_SCHEMA_VERSION
validation_dataset['source'] = 'derived_receipt_time_boundary_research_fixture'
validation_dataset['record_id'] = [
    hashlib.sha256(f'{VALIDATION_SCHEMA_VERSION}|{RUN_ID}|{symbol}|{time.isoformat()}|{settings_fingerprint}'.encode()).hexdigest()
    for symbol, time in zip(validation_dataset.streamer_symbol, validation_dataset.boundary_utc)
]
validation_dataset['market_time_synchronization_status'] = 'unproven_quote_source_times_unknown'
validation_dataset['numerical_bsm_eligibility_status'] = 'not_evaluated'
validation_dataset['source_time_unknown_reason'] = 'Quote source bid/ask times unavailable; receipts are not source times'
validation_dataset['selection_uses_after_boundary_data'] = True
assert len(validation_dataset) == len(boundary_candidates)
assert validation_dataset.record_id.is_unique
print(f'Prepared {len(validation_dataset):,} windows; {validation_dataset.exploratory_candidate.sum():,} candidate flags.')
print('All rows retained. Source database required for analytics/candles/raw provenance; no BSM computations performed.')
display(validation_dataset[['record_id','streamer_symbol','boundary_utc','exploratory_candidate',
                            'option_bid','option_ask','spy_bid','spy_ask','failed_checks']].head(10))


In [ ]:
WRITE_VALIDATION_EXPORT = False
EXPORT_DIRECTORY = repo_root / 'Tastyware_Demo_Scripts' / 'exports' / 'validation' / f'bsm_validation_v1_{RUN_ID}'

def write_validation_export():
    from datetime import datetime, timezone
    import os
    import shutil
    import tempfile

    target = EXPORT_DIRECTORY.resolve()
    ignored_root = (repo_root / 'Tastyware_Demo_Scripts' / 'exports').resolve()
    if not target.is_relative_to(ignored_root) or target == ignored_root:
        raise ValueError('Export must be a named subdirectory of the ignored exports directory.')
    if target.exists():
        raise FileExistsError(f'Export already exists; choose a new directory/version: {target}')

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024*1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    # A stopped, checkpointed capture is needed for a stable database fingerprint.
    database_wal = Path(str(DATABASE_PATH)+'.wal')
    if database_wal.exists():
        raise RuntimeError('Source WAL exists; stop/checkpoint the collector before export.')
    source_before = DATABASE_PATH.stat()
    database_hash = sha256_file(DATABASE_PATH)
    run_metadata = query('SELECT * FROM runs WHERE run_id=?', [RUN_ID])
    if len(run_metadata) != 1 or pd.isna(run_metadata.iloc[0]['ended_at']):
        raise ValueError('Choose one ended run before exporting.')
    source_nb = json.loads((repo_root/'Tastyware_Demo_Scripts'/'14_explore_spy_capture.ipynb').read_text(encoding='utf-8'))
    limitations_markdown = ''.join(next(c['source'] for c in source_nb['cells'] if c.get('id')=='preliminary-handoff'))
    target.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix='bsm_validation_staging_', dir=target.parent))
    try:
        parquet_path = stage/'validation_windows.parquet'
        with duckdb.connect(':memory:') as export_connection:
            export_connection.register('validation_rows', validation_dataset)
            export_connection.execute(
                'COPY validation_rows TO ? (FORMAT PARQUET, COMPRESSION ZSTD)', [str(parquet_path)]
            )
            saved = export_connection.execute('SELECT * FROM read_parquet(?)', [str(parquet_path)]).df()
            pd.testing.assert_frame_equal(
                validation_dataset.sort_values('record_id').reset_index(drop=True),
                saved.sort_values('record_id').reset_index(drop=True),
                check_dtype=False,
            )
            schema_rows = export_connection.execute('DESCRIBE SELECT * FROM read_parquet(?)', [str(parquet_path)]).df()
        schema = schema_rows[['column_name','column_type','null']].to_dict('records')
        metadata_json = json.loads(run_metadata.to_json(orient='records',date_format='iso',date_unit='us'))[0]
        manifest = {
            'validation_schema_version': VALIDATION_SCHEMA_VERSION,
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'dataset_role': 'preliminary_research_fixture_not_strict_handoff_acceptance',
            'self_contained': False,
            'source_database': {'path': str(DATABASE_PATH.resolve()), 'sha256': database_hash,
                                'bytes': source_before.st_size, 'required': True},
            'run': metadata_json,
            'selection_settings': validation_settings,
            'settings_sha256': settings_fingerprint,
            'counts': {'windows': len(validation_dataset),
                       'candidate_windows': int(validation_dataset.exploratory_candidate.sum()),
                       'contracts': int(validation_dataset.streamer_symbol.nunique()),
                       'boundaries': int(validation_dataset.boundary_utc.nunique()),
                       'rows_without_before_boundary_pair': int(validation_dataset.option_observation_id.isna().sum())},
            'files': {'validation_windows.parquet': {'sha256': sha256_file(parquet_path),
                                                     'rows': len(saved), 'schema': schema}},
            'null_semantics': 'Null means unavailable/undefined, not zero; see source_time_unknown_reason and failed_checks.',
            'analytics_and_candles': 'Remain in source DuckDB; no automatic alignment or model-input substitution.',
            'candidate_flag': 'Illustrative retrospective stability screen; not numerical BSM or source-time eligibility.',
            'limitations_markdown': limitations_markdown,
            'handoff_notes': heading_notes,
        }
        (stage/'manifest.json').write_text(json.dumps(manifest,indent=2,allow_nan=False),encoding='utf-8')
        source_after = DATABASE_PATH.stat()
        if database_wal.exists() or (source_before.st_size,source_before.st_mtime_ns) != (source_after.st_size,source_after.st_mtime_ns):
            raise RuntimeError('Source database changed during export; export refused.')
        # Rename only the new staging directory; never overwrite an existing export.
        os.rename(stage, target)
    except BaseException:
        shutil.rmtree(stage)
        raise
    print(f'Exported verified Parquet and manifest: {target}')
    print(f'Rows: {len(saved):,}; candidates: {int(saved.exploratory_candidate.sum()):,}; source database unchanged.')
    return target

heading_notes = 'Shared-database fixture: all windows retained; original IDs resolve in the source capture. Not a self-contained collector export. Rates/dividends and numerical BSM acceptance are downstream decisions.'
if WRITE_VALIDATION_EXPORT:
    VALIDATION_EXPORT_PATH = write_validation_export()
else:
    print('Export disabled. Set WRITE_VALIDATION_EXPORT=True to write the first version; existing directories are never overwritten.')


## Next: turn exploration into handoff checks

This first pass establishes **what is present**, not whether it is sufficient for BSM validation. We can incrementally add:

1. Per-instrument coverage, update frequencies, sequence integrity and reconnect gaps.
2. Missingness and quote quality distributions; explicit receipt-time option/SPY alignment experiments.
3. Mapping of contract terms, IV/Greek units, candle revisions and provenance to each handoff requirement.
4. The external rate/dividend/model assumptions needed before numerical BSM comparisons.

References:
- [Collector operator and schema guide](https://github.com/khazaddim/Tasty_Dearcyfi_Tests/blob/main/Tastyware_Demo_Scripts/README.md)
- [BSM data-collection handoff](https://github.com/khazaddim/Macro_Ideas/blob/main/short_put_replay/quote_collector_requirements.md)
- [BSM implementation](https://github.com/khazaddim/Macro_Ideas/blob/main/short_put_replay/bsm_model.py)

No API credentials or network requests are needed by this notebook. Preserve the database and WAL together if moving a stopped capture, and do not commit captured data or saved private notebook outputs.